# Fractal and Volatility Features: Main Workflow

This notebook orchestrates the entire workflow for the fractal and volatility feature analysis. It uses the modularized code from the `src` directory to ensure a clean, reusable, and maintainable process.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.config import default_config
from src.data_loader import load_price_data
from src.feature_engineering import create_features
from src.labeling import create_labels
from src.modeling import run_walk_forward
from src.backtesting import run_backtests
from src.diagnostics import (
    plot_reliability_diagram,
    plot_permutation_importance,
    plot_equity_curves,
    generate_summary_report
)

# Set a random seed for reproducibility
np.random.seed(default_config.random_seed)

## 1. Load Configuration

In [ ]:
config = default_config
print(f"Loaded configuration for ticker: {config.ticker}")

## 2. Load Data

In [ ]:
prices_df = load_price_data(config)

## 3. Feature Engineering

In [ ]:
features_df = create_features(prices_df, config)

## 4. Labeling

In [ ]:
labels = create_labels(features_df, config)

## 5. Align Data and Prepare for Model

In [ ]:
data_aligned = pd.concat([features_df, labels], axis=1).dropna()
X = data_aligned.drop(columns=['Date', 'Close', 'logret', 'target'])
y = data_aligned['target'].astype(int)
print(f"Final dataset shape: X={X.shape}, y={y.shape}")

## 6. Run Walk-Forward Validation

In [ ]:
model_results = run_walk_forward(X, y, config)

## 7. Run Backtesting

In [ ]:
oof_predictions_with_dates = model_results['oof_predictions'].set_index(X.index[model_results['oof_predictions'].index]).reset_index()
backtest_results = run_backtests(
    oof_predictions_with_dates,
    prices_df,
    model_results['best_threshold'],
    config
)

## 8. Diagnostics and Reporting

In [ ]:
generate_summary_report(config, prices_df, X, y, model_results, backtest_results)

In [ ]:
plot_equity_curves(backtest_results, config)

In [ ]:
plot_reliability_diagram(model_results['oof_predictions']['y_true'], model_results['oof_predictions']['p_hat'])

In [ ]:
plot_permutation_importance(X, y, X.columns.tolist(), config)